# 02. Postpartum Blood Pressure Prototype

A first pass at the MVP's core logic on synthetic demo data. For each patient the notebook computes days since delivery, flags readings in the hypertensive range (at or above 140/90) and the severe range (at or above 160/110), and checks whether readings were captured in the two recommended follow-up windows: postpartum days 3 to 4 and days 7 to 10.

**Placeholder:** the Exchange does not hold delivery dates. Until the EHR link exists, the delivery date is set to two days before each patient's first reading. Replace `delivery_dates` with real values when available.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from jupyterhealth_client import JupyterHealthClient, Code

client = JupyterHealthClient()

study_id = 30001  # replace with a blood pressure study id from notebook 01

In [ ]:
raw = client.list_observations_df(study_id=study_id, code=Code.BLOOD_PRESSURE, limit=None)

bp = raw[[
    "subject_reference",
    "effective_time_frame_date_time",
    "systolic_blood_pressure_value",
    "diastolic_blood_pressure_value",
]].rename(columns={
    "subject_reference": "patient",
    "effective_time_frame_date_time": "time",
    "systolic_blood_pressure_value": "sbp",
    "diastolic_blood_pressure_value": "dbp",
})
bp.head()

In [ ]:
# Placeholder delivery dates: two days before each patient's first reading
delivery_dates = bp.groupby("patient")["time"].min() - pd.Timedelta(days=2)

bp["day_pp"] = (bp["time"] - bp["patient"].map(delivery_dates)).dt.days
bp["hypertensive"] = (bp["sbp"] >= 140) | (bp["dbp"] >= 90)
bp["severe"] = (bp["sbp"] >= 160) | (bp["dbp"] >= 110)
bp.head()

## Handoff summary: one row per patient

In [ ]:
handoff = bp.groupby("patient").agg(
    n_readings=("sbp", "size"),
    last_day_pp=("day_pp", "max"),
    mean_sbp=("sbp", "mean"),
    mean_dbp=("dbp", "mean"),
    n_hypertensive=("hypertensive", "sum"),
    n_severe=("severe", "sum"),
    window_d3_4=("day_pp", lambda d: d.between(3, 4).any()),
    window_d7_10=("day_pp", lambda d: d.between(7, 10).any()),
).round(1)
handoff

## Trajectory for one patient

In [ ]:
patient = handoff.index[0]
p = bp[bp["patient"] == patient]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p["day_pp"], p["sbp"], "o-", label="Systolic")
ax.plot(p["day_pp"], p["dbp"], "o-", label="Diastolic")
for y in (140, 90):
    ax.axhline(y, ls="--", lw=1, color="gray")
for y in (160, 110):
    ax.axhline(y, ls=":", lw=1, color="red")
ax.axvspan(3, 4, alpha=0.15)
ax.axvspan(7, 10, alpha=0.15)
ax.set_xlabel("Days postpartum (placeholder delivery date)")
ax.set_ylabel("mmHg")
ax.set_title(patient)
ax.legend()
plt.show()

## Next steps

1. Replace placeholder delivery dates with EHR-sourced dates (SMART on FHIR).
2. Add glucose logic for patients with gestational diabetes.
3. Move the handoff table and plot into a Voila dashboard modeled on the `demos` repository.